In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp

In [0]:
df=spark.read.format("delta")\
    .load("/Volumes/workspace/bronze/bronzevolume/flights/data/")
df=df.drop("_rescued_data")\
        .withColumn('modification',current_timestamp())
display(df)

In [0]:
@dp.view(
    name="trans_flights"
)
def trans_flights():

    df=spark.readStream.format("delta")\
        .load("/Volumes/workspace/bronze/bronzevolume/flights/data")
    return df

In [0]:
dp.create_streaming_table("silver_flights")

dp.create_auto_cdc_flow(
  target = "silver_flights",
  source = "trans_flights",
  keys = ["flight_id"],
  ssequence_by = col("flight_id"),
  stored_as_scd_type = 1
)